In [1]:
import os
from dataset_wo_config_files import  CDCR_Dataset
from sentence_transformers import SentenceTransformer, util
import torch

#Load data
ecb_paths = ['../CDCR/data/ECBplus-prep/output_data/train']
cdcr_train = CDCR_Dataset(name="cdcr_train", data_path=ecb_paths,context_method="hsu")


def remove_special_token(token_list):
    return [token for token in token_list if token not in ('[E]','[/E]')]

# Load SBERT model
model = SentenceTransformer('all-mpnet-base-v2')

c:\Users\Sergio\miniconda3\envs\mbib\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [18]:
topic_list = set([mention.topic_id for _,mention in cdcr_train.mentions.items()])
mention_types = set([mention.type for _,mention in cdcr_train.mentions.items()])

In [19]:
dict_positive_pairs = {mention_type: {} for mention_type in mention_types}
dict_negative_pairs = {mention_type: {} for mention_type in mention_types}


for topic in topic_list:
    for mention_type in mention_types:
        keys_current_topic= cdcr_train.mentions_by_topics[topic]
        #filter by mention type
        mentions_sentences_current_topic= {k: remove_special_token(mention.new_context[1])  for k,mention in cdcr_train.mentions.items() if k in keys_current_topic and mention.type==mention_type}
        if not mentions_sentences_current_topic:
            continue
        #concatenate tokens
        sentences_list = [" ".join(sentence) for sentence in list(mentions_sentences_current_topic.values())]
        print("topic",topic,"mention_type",mention_type,len(sentences_list))
        #Compute similarity
        sentence_embeddings = model.encode(sentences_list, convert_to_tensor=True)
        cosine_scores = util.cos_sim(sentence_embeddings, sentence_embeddings)

        list_of_keys = list(mentions_sentences_current_topic.keys())
        # Get the indices of the lower triangular elements
        indices = torch.tril(torch.ones_like(cosine_scores), diagonal=-1).nonzero()

        # Create the dictionary using the indices
        lower_triangle_dict = {(i.item(), j.item()): cosine_scores[i, j].item() for i, j in indices}

        #classify pairs in: pairs from the same sentence, positives and negatives
  
        positive_pairs = {}
        negative_pairs = {}
        for k,v in lower_triangle_dict.items():
            row, col = k
            mention1 = cdcr_train.mentions[list_of_keys[row]]
            mention2 = cdcr_train.mentions[list_of_keys[col]]
            pair_key = f"{mention1.mention_id}###{mention2.mention_id}"
            #Same sentence
            if mention1.conll_doc_key==mention2.conll_doc_key and mention1.sent_id==mention2.sent_id:
                pass
            elif mention1.coref_chain == mention2.coref_chain:
                positive_pairs[pair_key]=v
            else: 
                negative_pairs[pair_key] = v
        
        # Concatenate positive_pairs and negative_pairs to the respective dictionaries
        dict_positive_pairs[mention_type].update(positive_pairs)
        dict_negative_pairs[mention_type].update(negative_pairs)
        


topic 19 mention_type event 128
topic 19 mention_type entity 191
topic 20 mention_type event 92
topic 20 mention_type entity 145
topic 28 mention_type event 176
topic 28 mention_type entity 327
topic 14 mention_type event 66
topic 14 mention_type entity 97
topic 32 mention_type event 108
topic 32 mention_type entity 227
topic 3 mention_type event 97
topic 3 mention_type entity 239
topic 30 mention_type event 169
topic 30 mention_type entity 420
topic 9 mention_type event 53
topic 9 mention_type entity 167
topic 34 mention_type event 79
topic 34 mention_type entity 318
topic 16 mention_type event 60
topic 16 mention_type entity 133
topic 24 mention_type event 99
topic 24 mention_type entity 225
topic 22 mention_type event 134
topic 22 mention_type entity 226
topic 29 mention_type event 146
topic 29 mention_type entity 315
topic 27 mention_type event 143
topic 27 mention_type entity 416
topic 33 mention_type event 144
topic 33 mention_type entity 297
topic 10 mention_type event 45
topic 

In [21]:
import json

json_file_path = "positive_pairs.json"

# Export dictionary to JSON file
with open(json_file_path, 'w') as json_file:
    json.dump(dict_positive_pairs, json_file, indent=4)

json_file_path = "negative_pairs.json"

# Export dictionary to JSON file
with open(json_file_path, 'w') as json_file:
    json.dump(dict_negative_pairs, json_file, indent=4)